# Notebook 2 – Prepare a Genome Assembly for fDOG-Assembly

fDOG-Assembly searches for orthologs directly in **unannotated genome assemblies**.
The target assembly is a genomic FASTA file (nucleotide sequences) of the species
you want to search in.

Genome assemblies can be downloaded from NCBI, e.g. via the
[NCBI FTP server](https://ftp.ncbi.nlm.nih.gov/genomes/all/).
Place the downloaded `.fna.gz` file in the `data/` folder before running this notebook.

This notebook shows how to:
1. Decompress the downloaded assembly
2. Register it with `fdog.addAssembly`, which creates the required directory structure

---

**Example used here:** *Rattus norvegicus* (rat), assembly GRCr8
(`GCF_036323735.1_GRCr8_genomic.fna.gz`), NCBI Taxonomy ID 10116.

---

### Prerequisites
| Requirement | How to obtain |
|---|---|
| fDOG installed | `pip install fdog` |
| Assembly FASTA | downloaded from NCBI FTP and placed in `data/` |

## 1 – Configuration

In [3]:
import gzip
import shutil
import subprocess
from pathlib import Path

# ── Input assembly ────────────────────────────────────────────────────────────
# Path to the downloaded .fna.gz file (relative to this notebook)
ASSEMBLY_GZ = Path("data/GCF_036323735.1_GRCr8_genomic.fna.gz")

# NCBI Taxonomy ID of the assembly species
NCBI_TAXON_ID = "10116"   # Rattus norvegicus

# Version string – choose any short label (used in the species folder name)
ASSEMBLY_VERSION = "v1"

# ── Output ────────────────────────────────────────────────────────────────────
ASSEMBLY_DIR = Path("data/assembly_dir")
ASSEMBLY_DIR.mkdir(parents=True, exist_ok=True)

assert ASSEMBLY_GZ.exists(), f"Assembly file not found: {ASSEMBLY_GZ}"

print(f"Input file     : {ASSEMBLY_GZ}  ({ASSEMBLY_GZ.stat().st_size / 1e6:.0f} MB)")
print(f"NCBI Taxon ID  : {NCBI_TAXON_ID}")
print(f"Version        : {ASSEMBLY_VERSION}")
print(f"Output dir     : {ASSEMBLY_DIR}")

Input file     : data/GCF_036323735.1_GRCr8_genomic.fna.gz  (876 MB)
NCBI Taxon ID  : 10116
Version        : v1
Output dir     : data/assembly_dir


## 2 – Decompress the Assembly

`fdog.addAssembly` expects a plain (uncompressed) FASTA file.
We decompress the `.gz` file into the `data/` folder.

In [4]:
# Remove the .gz suffix to get the output path
assembly_fna = Path("data") / ASSEMBLY_GZ.stem  # e.g. data/GCF_..._genomic.fna

if assembly_fna.exists():
    print(f"Already decompressed: {assembly_fna}")
else:
    print(f"Decompressing {ASSEMBLY_GZ.name} ...")
    with gzip.open(ASSEMBLY_GZ, "rb") as f_in, open(assembly_fna, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    print(f"Done: {assembly_fna}  ({assembly_fna.stat().st_size / 1e6:.0f} MB)")

Decompressing GCF_036323735.1_GRCr8_genomic.fna.gz ...
Done: data/GCF_036323735.1_GRCr8_genomic.fna  (2885 MB)


## 3 – Register the Assembly with `fdog.addAssembly`

`fdog.addAssembly` creates the directory structure required by fDOG-Assembly:

```
assembly_dir/
└── ABBR@TAXID@VERSION/
    └── ABBR@TAXID@VERSION.fa
```

The species abbreviation (`ABBR`) is derived automatically from the NCBI taxonomy
name of the provided taxon ID.

In [5]:
cmd = [
    "fdog.addAssembly",
    "--fasta", str(assembly_fna),
    "--out",   str(ASSEMBLY_DIR),
    "--ncbi",  NCBI_TAXON_ID,
    "--ver",   ASSEMBLY_VERSION,
]

print("Running:", " ".join(cmd))
print()
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError(f"fdog.addAssembly failed (exit {result.returncode})")

Running: fdog.addAssembly --fasta data/GCF_036323735.1_GRCr8_genomic.fna --out data/assembly_dir --ncbi 10116 --ver v1

#################################
NCBI taxon info: 10116 Rattus norvegicus
DONE, files can be found: /home/hannah/Dev/fdog-assembly/fDOG-Assembly/examples/data/assembly_dir/



## 4 – Verify the Output

In [6]:
print(f"Contents of {ASSEMBLY_DIR}:")
ASSEMBLY_SPECIES = None
for sp_dir in sorted(ASSEMBLY_DIR.iterdir()):
    if not sp_dir.is_dir():
        continue
    fasta = sp_dir / f"{sp_dir.name}.fa"
    if fasta.exists():
        print(f"  {sp_dir.name}/")
        print(f"    {fasta.name}  ({fasta.stat().st_size / 1e6:.0f} MB)")
        ASSEMBLY_SPECIES = sp_dir.name
    else:
        print(f"  {sp_dir.name}/  <- FASTA missing")

if ASSEMBLY_SPECIES:
    print()
    print("Use in fdog.assembly (Notebook 3):")
    print(f"  --assemblyPath {ASSEMBLY_DIR}")
    print(f"  --searchTaxa   {ASSEMBLY_SPECIES}")

Contents of data/assembly_dir:
  RATNO@10116@v1/
    RATNO@10116@v1.fa  (2885 MB)

Use in fdog.assembly (Notebook 3):
  --assemblyPath data/assembly_dir
  --searchTaxa   RATNO@10116@v1


## Summary

| Step | Result |
|---|---|
| Decompress | `data/GCF_036323735.1_GRCr8_genomic.fna` |
| `fdog.addAssembly` | `data/assembly_dir/RAT@10116@v1/RAT@10116@v1.fa` |

**Next step:** `03_run_fdog_assembly.ipynb` – run fDOG-Assembly to find the
ortholog in the rat genome.